## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [3]:
import pandas as pd
import seaborn as sns


titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
419,0,3,female,10.0,0,2,24.1500,S,Third,child,False,NaN,Southampton,no,False
860,0,3,male,41.0,2,0,14.1083,S,Third,man,True,NaN,Southampton,no,False
79,1,3,female,30.0,0,0,12.4750,S,Third,woman,False,NaN,Southampton,yes,True
738,0,3,male,NaN,0,0,7.8958,S,Third,man,True,NaN,Southampton,no,True
631,0,3,male,51.0,0,0,7.0542,S,Third,man,True,NaN,Southampton,no,True


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [ ]:
titanic_data.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [ ]:
n_rows = len(titanic_data)
nan_per_col = titanic_data.isna().sum()
titanic_data = titanic_data.loc[:, nan_per_col <= n_rows / 2]

n_cols = titanic_data.shape[1]
nan_per_row = titanic_data.isna().sum(axis=1)
titanic_data = titanic_data.loc[nan_per_row <= n_cols / 2]

titanic_data.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
embark_town      2
alive            0
alone            0
dtype: int64

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [16]:
for who in ["man", "woman", "child"]:
    mask_who = titanic_data["who"] == who
    median_age = round(titanic_data.loc[mask_who, "age"].median())
    titanic_data.loc[mask_who, "age"] = titanic_data.loc[mask_who, "age"].fillna(median_age)

titanic_data['age'].isna().sum()

np.int64(0)

### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [ ]:
nan_per_row = titanic_data.isna().sum(axis=1)
titanic_data = titanic_data.loc[nan_per_row <= 1]

titanic_data.isna().sum().sum()

np.int64(0)

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [8]:
most_popular_city = titanic_data["embark_town"].value_counts().index[0]
most_popular_city

'Southampton'

### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [9]:
survival_rate = round(titanic_data["survived"].mean() * 100, 2)
survival_rate

np.float64(38.25)

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [10]:
survivors_by_town = titanic_data[titanic_data["survived"] == 1]["embark_town"].value_counts()
survivors_by_town

embark_town
Southampton    217
Cherbourg       93
Queenstown      30
Name: count, dtype: int64

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [12]:
classes = titanic_data["class"].unique()
survival_by_class = pd.Series(
    {
        cls: round(titanic_data.loc[titanic_data["class"] == cls, "survived"].mean() * 100, 2)
        for cls in classes
    }
)
survival_by_class

Third     24.24
First     62.62
Second    47.28
dtype: float64

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [13]:
rich = titanic_data[titanic_data["fare"] >= 100]
rich_survival_rate = round(rich["survived"].mean() * 100, 2)
rich_survival_rate

np.float64(73.58)

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [ ]:
children_alone = titanic_data[(titanic_data["who"] == "child") and (titanic_data["alone"] == True)]
len(children_alone)

6

Какие выводы вы можете сделать о выживших пассажирах Титаника? 